[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch11-hw-acceleration/01-mo-hinh-roofline.ipynb)

# Mô hình roofline: hai trần, điểm gãy và chỗ đứng của từng phép toán

Chương 11 trả lời câu hỏi "một phép toán chạy nhanh được tới đâu trên một bộ tăng tốc" bằng một
công thức một dòng, phương trình 2 của sách:

$$R_{attain} = \min\left(R_{peak},\ \mathrm{BW} \times I\right)$$

Hiệu năng đạt được là giá trị nhỏ hơn giữa thông lượng đỉnh $R_{peak}$ và tích của băng thông
(bandwidth) với **mật độ tính toán** (arithmetic intensity) $I$, tức số phép toán chia cho số
byte đi qua giao diện bộ nhớ. Sổ tay này dùng đúng công thức ấy với các con số phần cứng của
sách. Không có gì được đo ở đây: sổ tay 02 mới đo trên máy của bạn.

**Bạn sẽ làm:**
1. dựng roofline của V100, A100, H100 và tính ba điểm gãy (ridge point);
2. đếm FLOP và byte của các phép toán trong chương từ hình dạng tensor, để ra mật độ tính toán;
3. đặt chúng lên roofline của A100 và đọc ra phép nào nghẽn ở bộ nhớ (memory-bound);
4. xem batch đẩy một tầng dày đặc (dense layer) từ vùng này sang vùng kia;
5. làm lại các ngộ nhận cuối chương và bài kiểm tra khả thi của sách.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Hai trần và điểm gãy

Mô hình roofline (roofline model) có hai trần. Khi mật độ thấp, hiệu năng tăng tuyến tính theo
mật độ với độ dốc là băng thông: đó là vùng nghẽn ở bộ nhớ. Khi mật độ đủ cao, hiệu năng bão hoà
ở thông lượng đỉnh: vùng nghẽn ở tính toán (compute-bound). Hai trần gặp nhau ở **điểm gãy**,
$I_{ridge} = R_{peak} / \mathrm{BW}$, cũng là cân bằng phần cứng (hardware balance) của thiết bị.

Đơn vị khớp nhau mà không cần đổi: TB/s nhân FLOP/byte ra TFLOP/s.

**Dự đoán:** từ V100 (2017) tới H100 (2022), thông lượng đỉnh FP16 tăng gần 8 lần. Điểm gãy tăng
theo, giảm đi, hay giữ nguyên? Hãy đoán trước khi chạy ô dưới.

In [ ]:
# Bảng 12 của sách: đỉnh FP16 với Tensor Core (TFLOP/s) và băng thông bộ nhớ (TB/s).
GPU = {
    "V100": (sach(125, nguon="125 TFLOP/s · 0.9 TB/s"), sach(0.9, nguon="125 TFLOP/s · 0.9 TB/s")),
    "A100": (sach(312, nguon="312 TFLOP/s · 2.04 TB/s"), sach(2.04, nguon="312 TFLOP/s · 2.04 TB/s")),
    "H100": (sach(989, nguon="989 TFLOP/s · 3.35 TB/s"), sach(3.35, nguon="989 TFLOP/s · 3.35 TB/s")),
}


def tran(I, R_peak, BW):
    """Phương trình 2: hiệu năng đạt được (TFLOP/s) = min(R_peak, BW × I)."""
    return np.minimum(R_peak, BW * np.asarray(I, dtype=float))


def diem_gay(R_peak, BW):
    """Mật độ tại đó hai trần bằng nhau (FLOP/byte)."""
    return R_peak / BW


for ten, (R, BW) in GPU.items():
    print(f"{ten}: {R:>4} TFLOP/s, {BW:>4} TB/s → điểm gãy {diem_gay(R, BW):6.1f} FLOP/byte")
theo_sach("điểm gãy V100", diem_gay(*GPU["V100"]), sach=138.9, nguon="138.9 FLOP/byte")
theo_sach("điểm gãy A100", diem_gay(*GPU["A100"]), sach=153, nguon="153 FLOP/byte")
theo_sach("điểm gãy H100", diem_gay(*GPU["H100"]), sach=295.2, nguon="295.2 FLOP/byte")

# Napkin Math 1.3: hai tỉ lệ giữa H100 và A100.
ti_le_bw = GPU["H100"][1] / GPU["A100"][1]
ti_le_dinh = GPU["H100"][0] / GPU["A100"][0]
theo_sach("băng thông H100 / A100", ti_le_bw, sach=1.6, nguon="1.6× · 3.2×")
theo_sach("đỉnh tính toán H100 / A100", ti_le_dinh, sach=3.2, nguon="1.6× · 3.2×")

Điểm gãy tăng, vì sức tính tăng nhanh hơn băng thông: mái của H100 cao hơn A100 khoảng 3.2 lần,
còn độ dốc chỉ tăng khoảng 1.6 lần. Đường tensor càng mới thì một kernel càng phải dùng lại dữ
liệu nhiều hơn mới chạm được trần tính toán. Hình dưới vẽ ba roofline trên trục log, cả hai trục.

In [ ]:
MAU = {"V100": "#2a78d6", "A100": "#eb6834", "H100": "#1baf7a"}
I_truc = np.logspace(-1, np.log10(3000), 400)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
for ten, (R, BW) in GPU.items():
    ax.loglog(I_truc, tran(I_truc, R, BW), color=MAU[ten], lw=2, label=f"{ten}: {R} TFLOP/s, {BW} TB/s")
    g = diem_gay(R, BW)
    ax.plot([g], [R], "o", color=MAU[ten], ms=8, mec="white", mew=1.5)
    ax.annotate(f"điểm gãy {g:.1f}", (g, R), xytext=(6, -14), textcoords="offset points",
                fontsize=8, color="0.25")
ax.set_xlabel("mật độ tính toán I (FLOP/byte)")
ax.set_ylabel("hiệu năng đạt được (TFLOP/s)")
ax.set_title("Roofline của ba thế hệ GPU: điểm gãy dịch sang phải")
ax.set_ylim(0.05, 3000)
ax.legend(loc="lower right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 2. Một kernel nằm giữa hai điểm gãy

Napkin Math 1.3 giả sử một kernel có mật độ 200 FLOP/byte, giữ nguyên cách hiện thực khi chuyển
từ A100 sang H100.

**Dự đoán:** nó nhanh hơn bao nhiêu lần trên H100: gần 3.2 lần (tỉ lệ đỉnh tính toán), gần 1.6
lần (tỉ lệ băng thông), hay một số khác?

In [ ]:
I_k = sach(200, nguon="200 FLOP/byte")
r_a100 = float(tran(I_k, *GPU["A100"]))
r_h100 = float(tran(I_k, *GPU["H100"]))
print(f"A100: {r_a100:.0f} TFLOP/s ({'nghẽn ở tính toán' if I_k > diem_gay(*GPU['A100']) else 'nghẽn ở bộ nhớ'})")
print(f"H100: {r_h100:.0f} TFLOP/s ({'nghẽn ở tính toán' if I_k > diem_gay(*GPU['H100']) else 'nghẽn ở bộ nhớ'})")
print(f"mức lợi lý tưởng: {r_h100 / r_a100:.2f} lần, so với {ti_le_bw:.2f} (băng thông) và {ti_le_dinh:.2f} (đỉnh)")

Kernel chạm mái phẳng của A100 nhưng rơi xuống dốc của H100, nên mức lợi là
$3.35 \times 200 / 312 \approx 2.15$ lần. Sách viết rằng mức lợi lý tưởng "gần với tỉ lệ băng
thông 1.6× hơn là tỉ lệ đỉnh 3.2×", và phép tính cho đúng như vậy, dù không trùng hẳn với 1.6:
trên A100 kernel này còn nghẽn ở tính toán, nên tỉ lệ thật là trần mới của bộ nhớ chia cho trần cũ
của tính toán. Chỉ khi kernel nghẽn ở bộ nhớ trên **cả hai** chip thì mức lợi mới đúng bằng tỉ lệ
băng thông.

## 3. Đếm FLOP và byte từ hình dạng tensor

Mật độ tính toán không phải một thuộc tính bí ẩn: nó là hai phép đếm chia cho nhau. Phần này đếm
lại các ví dụ của chương, mỗi ví dụ theo đúng mô hình lưu lượng sách nêu: đọc mỗi đầu vào và
trọng số một lần, ghi mỗi đầu ra một lần. FP16 chiếm 2 byte, FP32 chiếm 4 byte, một phép nhân
cộng dồn (MAC) là 2 FLOP, và sách đếm MB là $10^6$ byte.

In [ ]:
FP16, FP32 = 2, 4
MB, GB = 1e6, 1e9

# Định nghĩa 1.5: ReLU đứng riêng ở FP32, đọc một lần và ghi một lần mỗi phần tử.
byte_relu = sach(8, nguon="1 phép toán mỗi 8 byte")
assert byte_relu == FP32 + FP32
theo_sach("mật độ của ReLU", 1 / byte_relu, sach=0.125, nguon="0.125")

# Phép nhân hai ma trận n × n chia khối tốt, FP16: đọc A và B, ghi C, mỗi thứ một lần.
n = sach(1024, nguon="1024 × 1024 · 341.3 FLOP/byte")
flop_mm = 2 * n**3
byte_mm = 3 * n * n * FP16
theo_sach("mật độ nhân ma trận 1024", flop_mm / byte_mm, sach=341.3, nguon="341.3 FLOP/byte")

ReLU làm một phép so sánh cho mỗi 8 byte, bất kể tensor lớn bao nhiêu. Phép nhân ma trận làm
$2n^3$ FLOP trên $3n^2$ phần tử, nên mật độ của nó là $n/3$ ở FP16 và **tăng theo kích thước**.
Đây là lý do một phép nhân lớn có thể nghẽn ở tính toán còn một phép theo từng phần tử thì không
bao giờ.

Giờ tới bốn ví dụ napkin math của chương, đếm từ hình dạng tensor.

In [ ]:
# Napkin Math 1.4: một tầng transformer, chiều ẩn 768, batch 32, độ dài chuỗi 512, FP16.
H = sach(768, nguon="768 · 32 · 512")
B = sach(32, nguon="768 · 32 · 512")
S = sach(512, nguon="768 · 32 · 512")
# Softmax: 12 đầu tập trung, 3 phép mỗi phần tử (mũ, cộng, chia) theo cách đếm của sách.
DAU = sach(12, trich=r"32 \(\times\) 12 \(\times\) 512 \(\times\) 512 \(\times\) 3")
PHEP_SOFTMAX = sach(3, trich=r"32 \(\times\) 12 \(\times\) 512 \(\times\) 512 \(\times\) 3")

qkv_flop = 2 * 3 * B * S * H * H                         # ba phép chiếu Q, K, V
qkv_byte = (B * S * H + 3 * H * H + 3 * B * S * H) * FP16  # đầu vào, trọng số, ba đầu ra
sm_flop = B * DAU * S * S * PHEP_SOFTMAX
sm_byte = B * DAU * S * S * FP16 * 2                       # đọc điểm số, ghi xác suất
theo_sach("FLOP phép chiếu QKV (GFLOP)", qkv_flop / GB, sach=58, nguon="58 GFLOP · 104.2 MB")
theo_sach("byte phép chiếu QKV (MB)", qkv_byte / MB, sach=104.2, nguon="58 GFLOP · 104.2 MB")
theo_sach("mật độ phép chiếu QKV", qkv_flop / qkv_byte, sach=556.4, nguon="556.4 FLOP/byte")
theo_sach("FLOP softmax (MFLOP)", sm_flop / MB, sach=302, nguon="302 MFLOP · 402.7 MB")
theo_sach("byte softmax (MB)", sm_byte / MB, sach=402.7, nguon="302 MFLOP · 402.7 MB")
theo_sach("mật độ softmax", sm_flop / sm_byte, sach=0.75, nguon="0.75 FLOP/byte")

In [ ]:
# Napkin Math 1.5: tích chập 3 × 3, đầu vào (32, 128, 56, 56), 256 kênh ra, FP16, giữ kích thước.
Nb = sach(32, nguon="32 · 128 · 56 × 56 · 256 · 3 × 3")
Cin = sach(128, nguon="32 · 128 · 56 × 56 · 256 · 3 × 3")
HW = sach(56, nguon="32 · 128 · 56 × 56 · 256 · 3 × 3")
Cout = sach(256, nguon="32 · 128 · 56 × 56 · 256 · 3 × 3")
K = sach(3, nguon="32 · 128 · 56 × 56 · 256 · 3 × 3")

so_dau_ra = Nb * Cout * HW * HW
flop_moi_dau_ra = Cin * K * K * 2
conv_flop = so_dau_ra * flop_moi_dau_ra
vao, trong_so, ra = Nb * Cin * HW * HW * FP16, Cout * Cin * K * K * FP16, so_dau_ra * FP16
conv_byte = vao + trong_so + ra
theo_sach("phần tử đầu ra", so_dau_ra, sach=25.7e6, nguon="25.7M · 2,304 · 59.2 GFLOP")
theo_sach("FLOP mỗi đầu ra", flop_moi_dau_ra, sach=2_304, nguon="25.7M · 2,304 · 59.2 GFLOP")
theo_sach("tổng FLOP (GFLOP)", conv_flop / GB, sach=59.2, nguon="25.7M · 2,304 · 59.2 GFLOP")
theo_sach("đầu vào (MB)", vao / MB, sach=25.7, nguon="25.7 MB · 0.6 MB · 51.4 MB · 77.7 MB")
theo_sach("trọng số (MB)", trong_so / MB, sach=0.6, nguon="25.7 MB · 0.6 MB · 51.4 MB · 77.7 MB")
theo_sach("đầu ra (MB)", ra / MB, sach=51.4, nguon="25.7 MB · 0.6 MB · 51.4 MB · 77.7 MB")
theo_sach("tổng lưu lượng (MB)", conv_byte / MB, sach=77.7, nguon="25.7 MB · 0.6 MB · 51.4 MB · 77.7 MB")
theo_sach("mật độ tích chập", conv_flop / conv_byte, sach=762.2, nguon="762.2 FLOP/byte")

In [ ]:
# Napkin Math 1.6: tầng dày đặc 2048 → 2048, batch 32, FP16.
D = sach(2048, nguon="2048 · 268.4 MFLOP · 8.4 MB · 8.7 MB · 131.1 KB")
Bd = sach(32, nguon="2048 · 1 · 32 · 256 · 31 · 204.8")
dense_flop = 2 * Bd * D * D
dense_io = Bd * D * FP16                     # đầu vào, và cũng là cỡ đầu ra
dense_byte = 2 * dense_io + D * D * FP16
theo_sach("FLOP tầng dày đặc (MFLOP)", dense_flop / MB, sach=268.4, nguon="268.4 MFLOP · 8.4 MB")
theo_sach("đầu vào hay đầu ra (KB)", dense_io / 1e3, sach=131.1, nguon="131.1 KB")
theo_sach("trọng số (MB)", D * D * FP16 / MB, sach=8.4, nguon="268.4 MFLOP · 8.4 MB · 8.7 MB")
theo_sach("tổng lưu lượng (MB)", dense_byte / MB, sach=8.7, nguon="8.4 MB · 8.7 MB")
theo_sach("mật độ tầng dày đặc", dense_flop / dense_byte, sach=31, nguon="31 FLOP/byte · 2,039 GB/s")

# Napkin Math 1.7: LayerNorm trên (32, 512, 768), khoảng 8 phép mỗi phần tử, FP16.
phan_tu_ln = sach(32, nguon="32 × 512 × 768") * sach(512, nguon="32 × 512 × 768") * sach(768, nguon="32 × 512 × 768")
PHEP_LN = sach(8, nguon="12.6M · 8 · 100.7 MFLOP")
ln_flop = phan_tu_ln * PHEP_LN
ln_tham_so = H * 2 * FP16                    # γ và β: 3,072 byte, không đáng kể
ln_byte = 2 * phan_tu_ln * FP16 + ln_tham_so
theo_sach("phần tử LayerNorm", phan_tu_ln, sach=12.6e6, nguon="12.6M · 8 · 100.7 MFLOP")
theo_sach("FLOP LayerNorm (MFLOP)", ln_flop / MB, sach=100.7, nguon="12.6M · 8 · 100.7 MFLOP")
theo_sach("đọc vào (MB)", phan_tu_ln * FP16 / MB, sach=25.2, nguon="50.3 MB · 25.2 MB")
theo_sach("tổng lưu lượng (MB)", ln_byte / MB, sach=50.3, nguon="100.7 MFLOP · 50.3 MB")
theo_sach("mật độ LayerNorm", ln_flop / ln_byte, sach=2.0, nguon="2.0 FLOP/byte · 77 lần")

# Napkin Math 1.9: GPT-2 XL sinh từng token, batch 1: mỗi token đọc lại mọi trọng số FP16.
tham_so_gpt2 = sach(1.5e9, nguon="1.5 tỷ · 3 GB · 3 GFLOP · 1 FLOP/byte")
gpt2_flop = 2 * tham_so_gpt2
gpt2_byte = tham_so_gpt2 * FP16
theo_sach("trọng số GPT-2 XL (GB)", gpt2_byte / GB, sach=3, nguon="1.5 tỷ · 3 GB")
theo_sach("FLOP mỗi token (GFLOP)", gpt2_flop / GB, sach=3, nguon="3 GB · 3 GFLOP")
theo_sach("mật độ khi giải mã", gpt2_flop / gpt2_byte, sach=1, nguon="3 GFLOP · 1 FLOP/byte")

## 4. Đặt các phép toán lên roofline của A100

**Dự đoán:** trong tám phép toán vừa đếm, bao nhiêu phép nghẽn ở bộ nhớ trên A100, nơi điểm gãy
là khoảng 153 FLOP/byte?

In [ ]:
R_A, BW_A = GPU["A100"]
PHEP = {  # tên: (FLOP, byte)
    "ReLU (FP32)": (1, byte_relu),
    "softmax": (sm_flop, sm_byte),
    "GPT-2 XL, batch 1": (gpt2_flop, gpt2_byte),
    "LayerNorm": (ln_flop, ln_byte),
    "tầng dày đặc, batch 32": (dense_flop, dense_byte),
    "nhân ma trận 1024": (flop_mm, byte_mm),
    "phép chiếu QKV": (qkv_flop, qkv_byte),
    "tích chập 3 × 3": (conv_flop, conv_byte),
}
I_phep = {ten: f / b for ten, (f, b) in PHEP.items()}
print(f"{'phép toán':<24}{'I (FLOP/byte)':>14}{'đạt được (TFLOP/s)':>21}{'% đỉnh':>9}  vùng")
for ten, I in I_phep.items():
    r = float(tran(I, R_A, BW_A))
    vung = "nghẽn ở tính toán" if I > diem_gay(R_A, BW_A) else "nghẽn ở bộ nhớ"
    print(f"{ten:<24}{I:>14,.3f}{r:>21,.2f}{100 * r / R_A:>8.1f}%  {vung}")

# Napkin Math 1.6 và 1.7 dùng 2,039 GB/s, cùng băng thông A100 viết đủ chữ số.
BW_A_GBs = sach(2_039, nguon="31 FLOP/byte · 2,039 GB/s")
r_dense = BW_A_GBs * I_phep["tầng dày đặc, batch 32"] / 1e3
r_ln = BW_A_GBs * I_phep["LayerNorm"] / 1e3
theo_sach("tầng dày đặc đạt được (TFLOP/s)", r_dense, sach=63.3, nguon="63.3 TFLOP/s · 20.3 phần trăm")
theo_sach("phần trăm đỉnh của tầng dày đặc", 100 * r_dense / R_A, sach=20.3, nguon="20.3 phần trăm")
theo_sach("LayerNorm đạt được (TFLOP/s)", r_ln, sach=4.1, nguon="4.1 TFLOP/s · 1 phần trăm")
theo_sach("phần trăm đỉnh của LayerNorm", 100 * r_ln / R_A, sach=1, nguon="4.1 TFLOP/s · 1 phần trăm")
r_gpt2 = float(tran(I_phep["GPT-2 XL, batch 1"], R_A, BW_A))
theo_sach("GPT-2 XL đạt được (TFLOP/s)", r_gpt2, sach=2.04, nguon="2.04 TFLOP/s · 0.7 phần trăm")
theo_sach("trần mức sử dụng của GPT-2 XL (%)", 100 * r_gpt2 / R_A, sach=0.7, nguon="2.04 TFLOP/s · 0.7 phần trăm")

Một chi tiết về làm tròn: sách nói LayerNorm thấp hơn điểm gãy **77 lần**. Lấy hai số sách đã làm
tròn, $153 / 2.0 = 76.5$, làm tròn lên thành 77. Tính với số chưa làm tròn thì ra khoảng 76.5,
cũng nằm đúng giữa 76 và 77, nên cả hai cách đều nhất quán với con số của sách.

In [ ]:
ridge_tron = sach(153, nguon="153 FLOP/byte")
I_ln_tron = sach(2.0, nguon="2.0 FLOP/byte · 77 lần")
theo_sach("LayerNorm thấp hơn điểm gãy (lần)", ridge_tron / I_ln_tron, sach=77, nguon="2.0 FLOP/byte · 77 lần")
print(f"không làm tròn: {diem_gay(R_A, BW_A) / I_phep['LayerNorm']:.2f} lần")

In [ ]:
# Định nghĩa 1.3: đường DRAM máy chủ minh hoạ, nếu trọng số phải đi qua nó thay vì bộ nhớ của GPU.
BW_host = sach(200, nguon="2.04 TB/s · 10.2 · 200 GB/s") / 1e3
theo_sach("băng thông A100 / DRAM máy chủ", BW_A / BW_host, sach=10.2, nguon="2.04 TB/s · 10.2")

LECH = {  # (dx, dy, căn lề) của nhãn quanh mỗi điểm, đổi bên xen kẽ để các nhãn không đè nhau
    "ReLU (FP32)": (10, -12, "left"), "softmax": (-8, 8, "right"), "GPT-2 XL, batch 1": (10, -12, "left"),
    "LayerNorm": (-8, 8, "right"), "tầng dày đặc, batch 32": (-8, 8, "right"),
    "nhân ma trận 1024": (-30, 18, "right"), "phép chiếu QKV": (0, 44, "center"), "tích chập 3 × 3": (0, 70, "center"),
}
fig, ax = plt.subplots(figsize=(8, 5))
I_ve = np.logspace(np.log10(0.05), 4, 400)
ax.loglog(I_ve, tran(I_ve, R_A, BW_A), color=MAU["A100"], lw=2, label="A100: HBM 2.04 TB/s")
ax.loglog(I_ve, tran(I_ve, R_A, BW_host), color="0.55", lw=1.5, ls="--",
          label="nếu dữ liệu đi qua DRAM máy chủ 200 GB/s")
ax.axvline(diem_gay(R_A, BW_A), color="0.8", lw=1, zorder=0)
for ten, I in I_phep.items():
    y = float(tran(I, R_A, BW_A))
    dx, dy, ha = LECH[ten]
    ax.plot([I], [y], "o", color="#2a78d6", ms=8, mec="white", mew=1.5, zorder=3)
    ax.annotate(f"{ten}: I = {I:,.3g}", (I, y), xytext=(dx, dy), textcoords="offset points", ha=ha,
                fontsize=7.5, color="0.2", arrowprops=dict(arrowstyle="-", color="0.6", lw=0.7))
ax.text(diem_gay(R_A, BW_A) * 1.08, 0.06, "điểm gãy 153", fontsize=8, color="0.4")
ax.set_xlabel("mật độ tính toán I (FLOP/byte)")
ax.set_ylabel("hiệu năng đạt được (TFLOP/s)")
ax.set_title("Các phép toán của chương trên roofline của A100")
ax.set_ylim(0.04, 1e4)
ax.set_xlim(0.05, 1e4)
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Năm phép toán nằm trên dốc, ba phép nằm trên mái. LayerNorm ra 1.3 phần trăm đỉnh, mà sách gọi
là "khoảng 1 phần trăm", và GPT-2 XL ra 0.65 phần trăm, sách làm tròn thành 0.7. Hãy để ý hai điều:

* Softmax và phép chiếu QKV thuộc **cùng một tầng transformer**, mà cách nhau gần ba bậc độ lớn
  về mật độ. Đây là lý do FlashAttention tập trung giảm lưu lượng bộ nhớ của attention chứ không
  giảm FLOP.
* Tầng dày đặc batch 32 và nhân ma trận 1024 đều là phép nhân ma trận. Khác biệt nằm ở chỗ tầng
  dày đặc chỉ dùng lại trọng số theo batch, nên mật độ của nó chỉ khoảng 31.

Đường xám là định nghĩa 1.3 của sách: nếu trọng số phải đi qua DRAM máy chủ 200 GB/s thay vì bộ
nhớ băng thông cao 2.04 TB/s, cả dốc tụt xuống 10.2 lần, và mọi phép bên trái điểm gãy chậm đi
đúng chừng ấy.

## 5. Batch đẩy một tầng dày đặc qua điểm gãy

Napkin Math 1.8 viết mật độ của tầng dày đặc FP16 $(B \times M) \times (M \times N)$ thành một hàm
của kích thước batch (batch size) $B$, phương trình 6:

$$I(B) = \frac{2BMN}{2BM + 2MN + 2BN}$$

Tử số là số FLOP, mẫu số là số byte: đọc đầu vào, đọc trọng số, ghi đầu ra, mỗi phần tử 2 byte.
Khi phần trọng số $2MN$ áp đảo, $I(B) \approx B$.

**Dự đoán:** với $M = N = 2048$, batch nhỏ nhất để tầng này vượt điểm gãy 153 của A100 là bao
nhiêu? Gần 153, hay lớn hơn hẳn?

In [ ]:
M = N = D


def mat_do_batch(B, M=M, N=N):
    return 2 * B * M * N / (2 * B * M + 2 * M * N + 2 * B * N)


theo_sach("mật độ ở batch 1", mat_do_batch(1), sach=1, nguon="2048 · 1 · 32 · 256 · 31 · 204.8")
theo_sach("mật độ ở batch 32", mat_do_batch(32), sach=31, nguon="2048 · 1 · 32 · 256 · 31 · 204.8")
theo_sach("mật độ ở batch 256", mat_do_batch(256), sach=204.8, nguon="2048 · 1 · 32 · 256 · 31 · 204.8")

ridge_A = diem_gay(R_A, BW_A)
B_vuot = next(b for b in range(1, 10_000) if mat_do_batch(b) > ridge_A)
print(f"batch nhỏ nhất vượt điểm gãy A100: {B_vuot} (mật độ {mat_do_batch(B_vuot):.1f})")
# Mật độ bão hoà dần: với B rất lớn, nó tiến tới MN / (M + N), không tiến tới B mãi.
print(f"giới hạn khi batch rất lớn: {M * N / (M + N):.0f} FLOP/byte")

Xấp xỉ $I \approx B$ chỉ đúng khi batch nhỏ so với $M$ và $N$. Khi batch lớn dần, hai số hạng
$2BM$ và $2BN$ không còn nhỏ, nên cần batch lớn hơn 153 mới vượt điểm gãy 153, và mật độ bão hoà
ở $MN/(M+N)$, tức 1,024 FLOP/byte cho tầng này.

Phần ngộ nhận cuối chương đặt cùng tầng này lên một GPU rẻ hơn, T4, có đỉnh Tensor Core FP16
65 TFLOP/s và điểm gãy 203.1 FLOP/byte. Băng thông của nó suy ra từ hai số ấy.

In [ ]:
R_T4 = sach(65, nguon="65 TFLOP/s · 203.1 FLOP/byte")
BW_T4 = R_T4 / sach(203.1, nguon="65 TFLOP/s · 203.1 FLOP/byte")
print(f"băng thông T4 suy ra: {BW_T4:.2f} TB/s")
theo_sach("A100 ở batch 1 (TFLOP/s)", float(tran(mat_do_batch(1), R_A, BW_A)), sach=2.04,
          nguon="2.04 TFLOP/s · 0.3 TFLOP/s")
theo_sach("T4 ở batch 1 (TFLOP/s)", float(tran(mat_do_batch(1), R_T4, BW_T4)), sach=0.3,
          nguon="2.04 TFLOP/s · 0.3 TFLOP/s")

cac_batch = 2 ** np.arange(0, 13)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(cac_batch, mat_do_batch(cac_batch), "o-", color="#2a78d6", ms=5)
ax1.axhline(ridge_A, color=MAU["A100"], lw=1.2, ls="--")
ax1.text(1.2, ridge_A * 1.2, "điểm gãy A100, 153", fontsize=8, color=MAU["A100"])
for b, (dx, dy, ha) in ((1, (8, 2, "left")), (32, (8, -14, "left")), (256, (-10, 8, "right"))):
    ax1.annotate(f"batch {b}: {mat_do_batch(b):.3g}", (b, mat_do_batch(b)), xytext=(dx, dy),
                 textcoords="offset points", fontsize=8, ha=ha)
ax1.set_ylabel("mật độ tính toán (FLOP/byte)")
ax1.set_title("Phương trình 6, M = N = 2048")

for ten, R, BW, mau in (("A100", R_A, BW_A, MAU["A100"]), ("T4", R_T4, BW_T4, "#4a3aa7")):
    ax2.plot(cac_batch, tran(mat_do_batch(cac_batch), R, BW), "o-", color=mau, ms=5,
             label=f"{ten}: đỉnh {R} TFLOP/s")
ax2.set_ylabel("hiệu năng đạt được (TFLOP/s)")
ax2.set_title("Cùng tầng ấy trên hai GPU")
ax2.legend(loc="lower right", fontsize=8, frameon=False)
for ax in (ax1, ax2):
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks([1, 4, 16, 64, 256, 1024, 4096], ["1", "4", "16", "64", "256", "1024", "4096"])
    ax.set_xlabel("kích thước batch B")
plt.tight_layout()
plt.show()

Ở batch 1, cả hai GPU đều nằm sâu trên dốc, và A100 dùng chưa tới 1 phần trăm sức tính của nó.
Phần sức tính đứng yên ấy là thứ bạn đã trả tiền mua. Vì vậy sách kết luận rằng một bộ tăng tốc
rẻ hơn có thể kinh tế hơn trong chế độ này, nhưng độ trễ, dung lượng và giá vẫn phải đo.

## 6. Các ngộ nhận cuối chương, tính lại

**Độ chính xác số.** Mỗi định dạng số có đường tính và đỉnh riêng, nên có điểm gãy riêng.

In [ ]:
R_A_fp32 = sach(19.5, nguon="19.5 TFLOP/s · 9.6 FLOP/byte")
theo_sach("điểm gãy A100 ở FP32", diem_gay(R_A_fp32, BW_A), sach=9.6, nguon="19.5 TFLOP/s · 9.6 FLOP/byte")
R_H_fp32 = sach(67, nguon="989 TFLOP/s · 67 TFLOP/s · 15×")
theo_sach("H100: FP16 tensor so với FP32 lõi CUDA", GPU["H100"][0] / R_H_fp32, sach=15,
          nguon="989 TFLOP/s · 67 TFLOP/s · 15×")
I_ln = I_phep["LayerNorm"]
print(f"LayerNorm (I = {I_ln:.1f}) trên A100: dưới điểm gãy FP16 {ridge_A:.0f}, "
      f"và vẫn dưới điểm gãy FP32 {diem_gay(R_A_fp32, BW_A):.1f}")

Dùng số FP32 cho một tải chạy BF16 làm mức sử dụng trông tệ vô cớ, và đặt kernel sai chỗ trên
roofline: một phép có $I = 31$ nghẽn ở tính toán trên đường FP32 của A100 nhưng nghẽn ở bộ nhớ
trên đường FP16.

**Phần cứng càng chuyên biệt không phải lúc nào cũng nhanh hơn.** Softmax của attention, với
mật độ 2 tới 5 FLOP/byte, trên A100 và trên một CPU có điểm gãy 10 tới 20 FLOP/byte:

In [ ]:
I_min, I_max = sach(2, nguon="2–5 FLOP/byte"), sach(5, nguon="2–5 FLOP/byte")
theo_sach("softmax trên A100, đầu thấp (TFLOP/s)", float(tran(I_min, R_A, BW_A)), sach=4.1, nguon="4.1–10.2 TFLOP/s")
theo_sach("softmax trên A100, đầu cao (TFLOP/s)", float(tran(I_max, R_A, BW_A)), sach=10.2, nguon="4.1–10.2 TFLOP/s")
theo_sach("mức sử dụng cao nhất (%)", 100 * float(tran(I_max, R_A, BW_A)) / R_A, sach=3.3, nguon="3.3 phần trăm")
# Một phép nghẽn ở bộ nhớ đạt I/I_ridge phần đỉnh: thấp nhất khi I nhỏ và điểm gãy cao.
cpu_lo, cpu_hi = sach(10, nguon="10–20 FLOP/byte · 10–50 phần trăm"), sach(20, nguon="10–20 FLOP/byte · 10–50 phần trăm")
theo_sach("phần đỉnh CPU, thấp nhất (%)", 100 * I_min / cpu_hi, sach=10, nguon="10–20 FLOP/byte · 10–50 phần trăm")
theo_sach("phần đỉnh CPU, cao nhất (%)", 100 * I_max / cpu_lo, sach=50, nguon="10–20 FLOP/byte · 10–50 phần trăm")

Cả hai thiết bị đều coi softmax là nghẽn ở bộ nhớ, nhưng CPU dùng được một phần lớn hơn hẳn đỉnh
(thấp hơn nhiều) của nó. **Bỏ qua băng thông khi chọn bộ tăng tốc** là cạm bẫy thứ nhất: một bộ
tăng tốc quảng cáo 300 TFLOP/s với 2 TB/s.

In [ ]:
R_q, BW_q = sach(300, nguon="300 TFLOP/s · 2 TB/s · 150 FLOP/byte"), sach(2, nguon="300 TFLOP/s · 2 TB/s")
theo_sach("điểm gãy", diem_gay(R_q, BW_q), sach=150, nguon="300 TFLOP/s · 2 TB/s · 150 FLOP/byte")
I_q = sach(1.5, nguon="1.5 FLOP/byte · 3 TFLOP/s · 1 phần trăm")
theo_sach("LayerNorm đạt được (TFLOP/s)", float(tran(I_q, R_q, BW_q)), sach=3, nguon="1.5 FLOP/byte · 3 TFLOP/s")
theo_sach("mức sử dụng (%)", 100 * float(tran(I_q, R_q, BW_q)) / R_q, sach=1, nguon="3 TFLOP/s · 1 phần trăm")

## 7. Bài kiểm tra khả thi của sách

Checkpoint 1.4 đề nghị bạn kiểm tra ba ràng buộc cứng trước khi mua phần cứng. Mỗi câu là một
phép chia, và ô dưới làm từng phép. Hãy tự trả lời trước khi chạy.

1. **Băng thông:** mô hình 70 tỷ tham số (140 GB) trên băng thông 1 TB/s, mỗi token mất bao lâu, so
   với mục tiêu 50 ms?
2. **Tính toán:** video 30 FPS cho mỗi lần suy luận bao nhiêu mi-li-giây?
3. **Roofline:** bộ tăng tốc 2 TFLOP/s và 200 GB/s có điểm gãy bao nhiêu, kernel 10 FLOP/byte nằm
   ở đâu, và xung nhịp nhanh hơn có giúp nó không?

In [ ]:
# 1. Mỗi token phải đọc mọi trọng số một lần: T_token = D_vol / BW.
D_vol = sach(140, nguon="70 tỷ tham số · 140 GB · 1 TB/s · 50 ms") * GB
BW_c = sach(1, nguon="140 GB · 1 TB/s") * 1e12
muc_tieu_ms = sach(50, nguon="1 TB/s · 50 ms")
t_token_ms = D_vol / BW_c * 1e3
print(f"1. T_token = {t_token_ms:.0f} ms, mục tiêu {muc_tieu_ms} ms → "
      f"{'đạt' if t_token_ms <= muc_tieu_ms else f'chậm hơn {t_token_ms / muc_tieu_ms:.1f} lần'}")

# 2. Ngân sách thời gian của một khung hình.
fps = sach(30, nguon="30 FPS · 33.3 ms")
theo_sach("2. ngân sách mỗi khung hình (ms)", 1e3 / fps, sach=33.3, nguon="30 FPS · 33.3 ms")

# 3. Điểm gãy, rồi đặt kernel lên đó; "xung nhịp nhanh hơn" chỉ nâng R_peak.
R_c = sach(2, nguon="2 TFLOP/s · 200 GB/s · 10 FLOP/byte")
BW_c2 = sach(200, nguon="2 TFLOP/s · 200 GB/s") / 1e3
I_c = sach(10, nguon="200 GB/s · 10 FLOP/byte")
theo_sach("3. điểm gãy (FLOP/byte)", diem_gay(R_c, BW_c2), sach=10, nguon="2 TFLOP/s · 200 GB/s · 10 FLOP/byte")
for he_so in (1.0, 1.5, 2.0):
    print(f"   đỉnh × {he_so}: kernel đạt {float(tran(I_c, R_c * he_so, BW_c2)):.2f} TFLOP/s")

Kernel 10 FLOP/byte nằm **đúng tại** điểm gãy: hai trần bằng nhau. Xung nhịp nhanh hơn nâng mái
lên, nhưng kernel lập tức chạm dốc, nên nó không nhanh thêm chút nào. Chỉ thêm băng thông, hoặc
dùng lại dữ liệu nhiều hơn để tăng $I$, mới giúp được. Câu hỏi dung lượng (7 tỷ tham số, 14 GB
trọng số FP16 trên GPU 16 GB) để lại cho bạn ở phần *Thử thêm*.

## Thử thêm

1. Ở ô `ridge`, thay đỉnh của H100 bằng đỉnh FP8 dày đặc mà sách nêu, gần 2,000 TFLOP/s (các dấu
   ✗ khi đó là điều được chờ đợi). Điểm gãy mới là bao nhiêu, và kernel 200 FLOP/byte ở phần 2 giờ
   nhanh hơn A100 bao nhiêu lần?
2. Ở phần 5, đổi `M = N = D` thành một tầng hẹp hơn, chẳng hạn `M = N = 512`. Tầng ấy còn vượt
   được điểm gãy của A100 ở batch nào không? Vì sao giới hạn $MN/(M+N)$ quyết định câu trả lời?
3. Câu dung lượng của checkpoint 1.4: 14 GB trọng số FP16 vừa GPU 16 GB, còn lại 2 GB. Viết một ô
   tính bộ nhớ đệm KV (KV cache) cần cho một chuỗi dài $L$ token, với số tầng, số đầu và chiều mỗi
   đầu do bạn giả định, rồi tìm $L$ lớn nhất còn vừa.

## Tóm lại

* Hiệu năng đạt được là $\min(R_{peak},\ \mathrm{BW} \times I)$: thấp hơn điểm gãy thì băng thông
  quyết định, cao hơn thì sức tính quyết định.
* Điểm gãy tăng qua các thế hệ GPU vì sức tính tăng nhanh hơn băng thông, nên kernel phải dùng lại
  dữ liệu nhiều hơn mới hưởng được chip mới.
* Mật độ tính toán là hai phép đếm: phép nhân ma trận có mật độ tăng theo kích thước, phép theo
  từng phần tử như ReLU, softmax, LayerNorm thì không, và chúng nghẽn ở bộ nhớ.
* Batch nâng mật độ của tầng dày đặc, nhưng mật độ bão hoà ở $MN/(M+N)$, và batch 1 để A100 dùng
  chưa tới 1 phần trăm sức tính.
* Đỉnh phải khớp với độ chính xác số mà tải thực sự dùng, nếu không kernel bị đặt sai vùng.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Tăng tốc phần cứng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch11-hw-acceleration/), dựng từ chương
[*Hardware Acceleration*](https://mlsysbook.ai/vol1/hw_acceleration/hw_acceleration.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.